In [7]:
!pip install addict accelerate rapidocr_onnxruntime

In [8]:
import os
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from rapidocr_onnxruntime import RapidOCR

# 1. Upload the invoice image
print("Please upload your invoice image:")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded. Please upload an image file.")

image_path = list(uploaded.keys())[0]
print(f"Uploaded successfully: {image_path}")

# 2. Extract Text using RapidOCR
print("Initializing RapidOCR...")
engine = RapidOCR()

print("Running RapidOCR on image...")
result, elapse = engine(image_path)

ocr_text = ""
if result:
    # Join detected lines of text together with newlines
    ocr_text = "\n".join([line[1] for line in result])

print("\n--- RapidOCR Extracted Text ---")
print(ocr_text)
print("-------------------------\n")

# 3. Structure the extracted details using Qwen model
print("Initializing Qwen model for structured extraction...")
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Construct an optimized prompt to get detailed structured metadata and multiple products
prompt = f"""You are an expert system designed to extract highly structured details from invoice OCR text.
Analyze the following OCR text and extract invoice information. Be precise and fast.

OCR Text:
{ocr_text}

Strictly return your final output in valid JSON format only, matching this structure exactly:
{{
  "invoice_metadata": {{
    "invoice_number": "<invoice number/ID>",
    "invoice_date": "<date of invoice>",
    "total_amount": "<total invoice amount>",
    "total_tax_value": "<total tax amount or text>"
  }},
  "products": [
    {{
      "raw_name": "<original product name from line item>",
      "family": "<product family/category e.g., TMT, Steel, etc.>",
      "grade": "<grade specification e.g., FE 550D>",
      "thickness": "<thickness dimension and unit e.g., 40mm>",
      "width": "<width if available>",
      "height": "<height if available>",
      "length": "<length if available e.g., 12.000mtr>",
      "product_code": "<material or service code>",
      "dimensions": "<any compiled dimension information>",
      "quantity": "<quantity/weight if available>",
      "value": "<item line value or taxable value>"
    }}
  ],
  "transportation_details": {{
    "transporter_name": "<name of transporter>",
    "eway_bill_no": "<e-way bill number>",
    "transporter_code": "<transporter ID/code>",
    "truck_vehicle_no": "<truck/vehicle number>",
    "lr_no": "<LR number>",
    "lr_date": "<LR date>"
  }}
}}
"""

messages = [
    {"role": "system", "content": "You are a precise JSON extractor. Output ONLY valid JSON starting with '{' and ending with '}' without markdown formatting blocks."},
    {"role": "user", "content": prompt}
]

text = qwen_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = qwen_tokenizer([text], return_tensors="pt").to(qwen_model.device)

print("Extracting structured details via Qwen...")
generated_ids = qwen_model.generate(
    **model_inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=False
)

generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
response = qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n--- Structured Invoice Details Output ---")
print(response)
print("-----------------------------------------")

Please upload your invoice image:


Saving ChatGPT Image Nov 2, 2026, 01_43_25 PM.jpg to ChatGPT Image Nov 2, 2026, 01_43_25 PM.jpg
Uploaded successfully: ChatGPT Image Nov 2, 2026, 01_43_25 PM.jpg
Initializing RapidOCR...
Running RapidOCR on image...

--- RapidOCR Extracted Text ---
IN
:L27100MH1907P-C0C0260
PAN:AAACT2803M
TATA STEELLIMITED
GSTIN:27AAACT2803M1ZB
Cold RollingMill,S/76,MDc Area.Tnrnput:401506,Mnharashtra, India
REGD.OFFICE·BOMBAYHOUSI.24HOMIMODYSTREET,FORT.MUMBAI-40O0O1
TAXINVOICE
ORIGINAL FOR RECIPILNI
LjotoDed
Doneste Sale lnvoice
Bin To
:163486
Ship To
:821033
Name
.GST Invoice No.
:7679033780
: Rohit & Company
Name
:Rchit&Company
So No.
Dale:12.11.2025
Address1
：9519257133
:B2-1003.marathen Inova,OppPennunsulaCorporate
: C/o Skm Metal Processus J/9: M.i.d.c.
Address 1
DO No.
Date12.11.2025
:9519258205
Park.
Taloja, Next To Fire
PO No.
Date:12.11.2025
Address 2
:Off.GaapetraoKadamMarg.lpisiB21o03.marthon
: Brigade,taloja Dist- Raigad J/93 M.i.d.c.
Address 2
:CRCW,Nov25Date:12.11.2025
Delivery No.
:9095

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Extracting structured details via Qwen...

--- Structured Invoice Details Output ---
```json
{
  "invoice_metadata": {
    "invoice_number": "7679033780",
    "invoice_date": "12.11.2025",
    "total_amount": "1,447,983.90",
    "total_tax_value": "1,447,983.90"
  },
  "products": [
    {
      "raw_name": "CRCOILTATXXD(IS513D)XA371.G0X1250",
      "family": "Steel",
      "grade": "FE 550D",
      "thickness": "40mm",
      "width": "",
      "height": "",
      "length": "12.000mtr",
      "product_code": "CRCOILTATXXD(IS513D)XA371.G0X1250",
      "dimensions": "40mm x 12.000mtr",
      "quantity": "",
      "value": "54.935.00"
    }
  ],
  "transportation_details": {
    "transporter_name": "SAIDATTALOGISrICSPRIVATE",
    "eway_bill_no": "2220753$6159",
    "transporter_code": "AADFR319OD",
    "truck_vehicle_no": "MH48CQ6892",
    "lr_no": "XN1911035134",
    "lr_date": "12.11.2025"
  }
}
```
-----------------------------------------
